# 🔍 Notebook 02 — Exa.ai Search Test

In [11]:

# Setup

import os
import json
from dotenv import load_dotenv
from exa_py import Exa  # Exa.ai client
from pprint import pprint


In [17]:
# 🔹 Enhanced Exa.ai search function
def search_exa(query, domains=None, top_k=5):
    """
    Perform semantic search on Exa.ai and return top sources.
    
    Args:
        query (str): The search query
        domains (list, optional): List of trusted domains (e.g., ["sae.org", "ieee.org"])
        top_k (int): Number of top sources to return
    
    Returns:
        list[dict]: List of top sources with title, url, snippet, score
    """
    try:
        # Call Exa.ai search
        response = exa.search(query=query)
        results = getattr(response, "results", [])
        
        if not results:
            print("⚠️ No results found for your query.")
            return []

        # Filter by domains if provided
        if domains:
            results = [r for r in results if any(d in getattr(r, "url", "") for d in domains)]
            if not results:
                print(f"⚠️ No results matched the trusted domains: {domains}")
        
        # Sort by relevance score descending (use 0 if score is None)
        results = sorted(results, key=lambda x: getattr(x, "score", 0) or 0, reverse=True)
        
        # Take top_k sources and ensure all fields are present
        top_sources = []
        for r in results[:top_k]:
            top_sources.append({
                "title": getattr(r, "title", "No title"),
                "url": getattr(r, "url", "No URL"),
                "snippet": getattr(r, "snippet", "No snippet available"),
                "score": getattr(r, "score", 0) or 0
            })
        
        return top_sources
    
    except Exception as e:
        print(f"❌ Error during Exa.ai search: {e}")
        return []


In [18]:
user_question = "Explain autonomous driving levels"
trusted_domains = ["sae.org", "ieee.org"]

top_sources = search_exa(user_question, domains=trusted_domains, top_k=5)

from pprint import pprint
print("🔹 Top Exa.ai sources:")
pprint(top_sources)

# Save results to JSON for backend use
import json
with open("exa_search_results.json", "w", encoding="utf-8") as f:
    json.dump(top_sources, f, ensure_ascii=False, indent=2)

print(f"✅ Top sources saved to exa_search_results.json")


🔹 Top Exa.ai sources:
[{'score': 0,
  'snippet': 'No snippet available',
  'title': 'SAE Levels of Driving Automation™ Refined for Clarity and ...',
  'url': 'https://www.sae.org/news/blog/sae-levels-driving-automation-clarity-refinements'}]
✅ Top sources saved to exa_search_results.json


In [19]:
user_question = "How do autonomous vehicles detect obstacles?"
top_sources = search_exa(user_question, top_k=5)
pprint(top_sources)


[{'score': 0,
  'snippet': 'No snippet available',
  'title': 'How Self-Driving Cars Detect and Avoid Obstacles: The Future of '
           '...',
  'url': 'https://skill-lync.com/blogs/how-self-driving-cars-detect-and-avoid-obstacles'},
 {'score': 0,
  'snippet': 'No snippet available',
  'title': 'LiDAR in Autonomous Vehicles: Transforming Navigation and Safety',
  'url': 'https://www.sapien.io/blog/lidar-in-autonomous-vehicles'},
 {'score': 0,
  'snippet': 'No snippet available',
  'title': 'Perception Technologies for Autonomous Transportation: A '
           'Comparative Analysis of LiDAR, Radar, Camera, and Sonar',
  'url': 'https://rosap.ntl.bts.gov/view/dot/87734/dot_87734_DS1.pdf'},
 {'score': 0,
  'snippet': 'No snippet available',
  'title': 'Object Detection in Autonomous Vehicles: Detailed Overview',
  'url': 'https://www.sapien.io/blog/object-detection-in-autonomous-vehicles'},
 {'score': 0,
  'snippet': 'No snippet available',
  'title': 'LiDAR and cameras in autonomous 

In [ ]:
# Realistic test question
user_question = "What are the main safety challenges for autonomous vehicles?"
trusted_domains = ["sae.org", "nhtsa.gov", "ieee.org"]

# Run the search
top_sources = search_exa(user_question, domains=trusted_domains, top_k=5)

# Display in a DataFrame
import pandas as pd
df_sources = pd.DataFrame(top_sources)
print("🔹 Top Exa.ai sources :")
display(df_sources)


🔹 Top Exa.ai sources for realistic query:


,title,url,snippet,score
0,Automated Vehicle Safety,https://www.nhtsa.gov/vehicle-safety/automated...,No snippet available,0


In [24]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 🔹 Function to fetch snippet from URL if missing
def fetch_snippet(url, max_chars=200):
    try:
        r = requests.get(url, timeout=5)
        soup = BeautifulSoup(r.text, 'html.parser')
        paragraphs = soup.find_all('p')
        for p in paragraphs:
            text = p.get_text().strip()
            if len(text) > 50 and "Reference" not in text:  # skip short or reference text
                return text[:max_chars] + "..."
    except:
        return "No snippet available"
    return "No snippet available"


# 🔹 Enhanced Exa.ai search function with snippet fallback
def search_exa(query, domains=None, top_k=5):
    """
    Perform semantic search on Exa.ai and return top sources with snippets.
    
    Args:
        query (str): The search query
        domains (list, optional): List of trusted domains (e.g., ["sae.org", "ieee.org"])
        top_k (int): Number of top sources to return
    
    Returns:
        list[dict]: List of top sources with title, url, snippet, score
    """
    try:
        response = exa.search(query=query)
        results = getattr(response, "results", [])

        if not results:
            print("⚠️ No results found for your query.")
            return []

        # Filter by domains if provided
        if domains:
            results = [r for r in results if any(d in getattr(r, "url", "") for d in domains)]
            if not results:
                print(f"⚠️ No results matched the trusted domains: {domains}")

        # Sort by relevance score descending
        results = sorted(results, key=lambda x: getattr(x, "score", 0) or 0, reverse=True)

        # Take top_k sources and fill missing snippet
        top_sources = []
        for r in results[:top_k]:
            title = getattr(r, "title", "No title")
            url = getattr(r, "url", "No URL")
            snippet = getattr(r, "snippet", "No snippet available")
            score = getattr(r, "score", 0) or 0

            # If snippet is missing, fetch from webpage
            if snippet == "No snippet available" and url != "No URL":
                snippet = fetch_snippet(url)

            top_sources.append({
                "title": title,
                "url": url,
                "snippet": snippet,
                "score": score
            })

        return top_sources

    except Exception as e:
        print(f"❌ Error during Exa.ai search: {e}")
        return []


In [25]:
# 1️⃣ Define your question
user_question = "What are the main safety challenges for autonomous vehicles?"
trusted_domains = ["sae.org", "nhtsa.gov", "ieee.org"]

# 2️⃣ Run the Exa.ai search
top_sources = search_exa(user_question, domains=trusted_domains, top_k=5)

# 3️⃣ Clean the snippet text
for src in top_sources:
    src['snippet'] = src['snippet'].replace("\n", " ").replace("\t", " ").strip()

# 4️⃣ Display as a DataFrame
import pandas as pd
df_sources = pd.DataFrame(top_sources)
print("🔹 Top Exa.ai sources with cleaned snippets:")
display(df_sources)


🔹 Top Exa.ai sources with cleaned snippets:


,title,url,snippet,score
0,Report to Congress: NHTSA Research and Rulemak...,https://www.nhtsa.gov/sites/nhtsa.gov/files/20...,https://errors.edgesuite.net/18.56ca3017.17684...,0


In [ ]:
user_question = "How do self-driving cars detect obstacles and pedestrians?"

# Run search without filtering domains
top_sources = search_exa(user_question, domains=None, top_k=5)

# Clean snippets
for src in top_sources:
    src['snippet'] = src['snippet'].replace("\n", " ").replace("\t", " ").strip()

# Display DataFrame
import pandas as pd
df_sources = pd.DataFrame(top_sources)
display(df_sources)

# Save results to JSON for backend use
import json
with open("exa_search_results.json", "w", encoding="utf-8") as f:
    json.dump(top_sources, f, ensure_ascii=False, indent=2)

print(f"✅ Top sources saved to exa_search_results.json")


,title,url,snippet,score
0,LiDAR in Autonomous Vehicles: Transforming Nav...,https://www.sapien.io/blog/lidar-in-autonomous...,Autonomous vehicles are transforming transport...,0
1,LiDAR and cameras in autonomous driving,https://www.nature.com/articles/s44287-025-001...,Thank you for visiting nature.com. You are usi...,0
2,"Sensor Suite: How Cameras, LiDAR, and RADAR Wo...",https://www.dpvtransportation.com/sensor-suite...,At the heart of every self-driving car is a po...,0
3,How Computer Vision Powers Autonomous Vehicles,https://www.labellerr.com/blog/how-self-drivin...,Data Annotation Platform C...,0
4,How Autonomous Vehicles Work: the Self-Driving...,https://www.mobileye.com/blog/autonomous-vehic...,Mobileye develops a full range of technologies...,0


In [29]:
import json
import pandas as pd
from IPython.display import display

# 1️⃣ Load top sources from JSON file
with open("exa_search_results.json", "r", encoding="utf-8") as f:
    top_sources = json.load(f)

# 2️⃣ Clean snippet text (remove newlines/tabs)
for src in top_sources:
    src['snippet'] = src['snippet'].replace("\n", " ").replace("\t", " ").strip()
    if not src['snippet']:
        src['snippet'] = "No snippet available"

# 3️⃣ User question
user_question = "How do self-driving cars detect obstacles and pedestrians?"

# 4️⃣ Placeholder AI answer (since Gemini quota is exceeded)
placeholder_answer = (
    "Self-driving cars detect obstacles and pedestrians using sensors like LiDAR, cameras, and radar, "
    "combined with AI perception algorithms to analyze the environment in real-time."
)

# 5️⃣ Combine into a table
combined_data = []
for src in top_sources:
    combined_data.append({
        "Question": user_question,
        "Source Title": src.get("title", ""),
        "URL": src.get("url", ""),
        "Snippet": src.get("snippet", ""),
        "AI Answer (placeholder)": placeholder_answer
    })

df_combined = pd.DataFrame(combined_data)

# 6️⃣ Display the table
print("🔹 Combined Question + Sources + AI Answer Table:")
display(df_combined)


🔹 Combined Question + Sources + AI Answer Table:


,Question,Source Title,URL,Snippet,AI Answer (placeholder)
0,How do self-driving cars detect obstacles and ...,LiDAR in Autonomous Vehicles: Transforming Nav...,https://www.sapien.io/blog/lidar-in-autonomous...,Autonomous vehicles are transforming transport...,Self-driving cars detect obstacles and pedestr...
1,How do self-driving cars detect obstacles and ...,LiDAR and cameras in autonomous driving,https://www.nature.com/articles/s44287-025-001...,Thank you for visiting nature.com. You are usi...,Self-driving cars detect obstacles and pedestr...
2,How do self-driving cars detect obstacles and ...,"Sensor Suite: How Cameras, LiDAR, and RADAR Wo...",https://www.dpvtransportation.com/sensor-suite...,At the heart of every self-driving car is a po...,Self-driving cars detect obstacles and pedestr...
3,How do self-driving cars detect obstacles and ...,How Computer Vision Powers Autonomous Vehicles,https://www.labellerr.com/blog/how-self-drivin...,Data Annotation Platform C...,Self-driving cars detect obstacles and pedestr...
4,How do self-driving cars detect obstacles and ...,How Autonomous Vehicles Work: the Self-Driving...,https://www.mobileye.com/blog/autonomous-vehic...,Mobileye develops a full range of technologies...,Self-driving cars detect obstacles and pedestr...


***********************************************
**trial**


In [ ]:
import requests
from bs4 import BeautifulSoup
from exa_py import Exa
from dotenv import load_dotenv
import os
import json


In [ ]:
load_dotenv()
EXA_API_KEY = os.getenv("EXA_API_KEY")

exa = Exa(api_key=EXA_API_KEY)


In [ ]:
def search_exa(query, domains=None, top_k=5):
    response = exa.search(query=query)
    results = getattr(response, "results", [])

    if not results:
        return []

    # Optional domain filtering
    if domains:
        results = [
            r for r in results
            if any(d in getattr(r, "url", "") for d in domains)
        ]

    # Sort by relevance
    results = sorted(
        results,
        key=lambda x: getattr(x, "score", 0) or 0,
        reverse=True
    )

    top_sources = []

    for r in results[:top_k]:
        title = getattr(r, "title", "No title")
        url = getattr(r, "url", "")
        snippet = getattr(r, "snippet", "") or "No snippet available"
        score = getattr(r, "score", 0)

        # 🔥 Snippet fallback
        if snippet == "No snippet available" and url:
            snippet = fetch_snippet(url)

        top_sources.append({
            "title": title,
            "url": url,
            "snippet": snippet,
            "score": score
        })

    return top_sources


In [ ]:
user_question = "How do self-driving cars detect obstacles and pedestrians?"

trusted_domains = [
    "ieee.org",
    "nature.com",
    "sae.org",
    "mobileye.com"
]

sources = search_exa(
    query=user_question,
    domains=trusted_domains,
    top_k=5
)

# Save for Gemini notebook
with open("exa_search_results.json", "w", encoding="utf-8") as f:
    json.dump(sources, f, indent=2, ensure_ascii=False)

print("✅ Exa results saved to exa_search_results.json")
